# Bootcamp pour GCH3545 
<p style="margin-top: -1.2em;">
Auteurs : Bruno Blais & Fabian Denner (Polytechnique Montréal)<br>
Licence : CC BY-SA 4.0
</p>

## Partie 5 – Tests et vérification

Dans les parties précédentes, nous avons :

- préparé l'environnement Python ;
- revu les bases utiles de Python ;
- construit un premier modèle numérique ;
- amélioré la qualité et la lisibilité du code.

Nous allons maintenant aborder une question centrale en modélisation numérique : **Comment savoir si un programme donne un résultat correct ?**

Cette question est particulièrement importante aujourd'hui, car il est très facile de générer du code rapidement avec des outils d'IA, de copier un exemple trouvé en ligne ou de réutiliser du code écrit par quelqu'un d'autre.

Mais une chose ne change pas :

##### **La responsabilité de vérifier un résultat appartient toujours à l'ingénieur, peu importe l'origine du code !**

Peu importe que le code ait été écrit :

- par vous ;
- par un collègue ;
- par un ancien étudiant ;
- par ChatGPT, Copilot, Claude ou un autre outil d'IA ;
- par une bibliothèque externe.

Un résultat n'est crédible que si vous avez des raisons de lui faire confiance.

## 🎯 Objectifs

À la fin de cette partie, vous devriez être capable de :

- expliquer pourquoi un code qui s'exécute n'est pas forcément correct ;
- formuler des vérifications simples à partir du comportement physique attendu ;
- utiliser `assert` pour automatiser une vérification ;
- écrire des tests simples avec `pytest` ;
- interpréter un test qui échoue ;
- comprendre pourquoi les tests facilitent la modification du code ;
- distinguer vérification et validation.

Cette partie prépare directement l'utilisation responsable des outils d'IA générative dans la Partie 6.

## 5.1 Préparation

Nous reprenons le modèle de refroidissement d'une sphère chaude :

$T(t) = T_\infty + (T_0 - T_\infty)\exp(-kt)$

### ▶ À faire

Exécutez la cellule suivante.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("✅ Bibliothèques importées")

Nous redéfinissons ici la fonction du modèle afin que cette partie du notebook puisse être exécutée de manière autonome. Donc, exécutez la cellule suivante.

In [ ]:
def temperature_sphere(t, temperature_initiale, temperature_ambiante, coefficient_refroidissement):
    """
    Calcule la température d'une sphère chaude en refroidissement.

    Paramètres
    ----------
    t : float ou ndarray
        Temps [s].
    temperature_initiale : float
        Température initiale de la sphère [°C].
    temperature_ambiante : float
        Température ambiante [°C].
    coefficient_refroidissement : float
        Coefficient de refroidissement [1/s].

    Retour
    ------
    float ou ndarray
        Température de la sphère [°C].
    """
    return (
        temperature_ambiante
        + (temperature_initiale - temperature_ambiante)
        * np.exp(-coefficient_refroidissement * t)
    )

---

## 5.2 Le code s'exécute : est-ce suffisant ?

Commençons par une fonction volontairement incorrecte.

Elle ressemble beaucoup au bon modèle, mais elle contient une erreur importante.

### ▶ À faire

Exécutez la cellule suivante.

In [ ]:
def temperature_sphere_fausse(t, temperature_initiale, temperature_ambiante, coefficient_refroidissement):
    """
    Version volontairement fausse du modèle de refroidissement.

    Attention : cette fonction contient une erreur.
    """
    return (temperature_ambiante + (temperature_initiale - temperature_ambiante) * np.exp(coefficient_refroidissement * t))

Utilisons cette fonction pour produire un graphique.

In [ ]:
temps = np.linspace(0, 200, 201)

temperature_initiale = 500.0
temperature_ambiante = 25.0
coefficient_refroidissement = 0.03

temperatures_fausses = temperature_sphere_fausse(temps, temperature_initiale, temperature_ambiante, coefficient_refroidissement)

plt.plot(temps, temperatures_fausses)
plt.xlabel("Temps [s]")
plt.ylabel("Température [°C]")
plt.title("Résultat produit par une fonction incorrecte")
plt.grid(True)
plt.show()

### ⚠ Réflexion

Le programme a-t-il produit une erreur Python ?

Probablement pas.

Le code s'est exécuté. Un graphique a été produit.

Mais le résultat est-il physiquement plausible ?

- La température devrait-elle augmenter ?
- La température devrait-elle devenir beaucoup plus grande que la température initiale ?
- Le comportement observé correspond-il à un refroidissement ?

C'est une leçon essentielle : **Un code qui s'exécute n'est pas nécessairement un code correct.**

Python vérifie la syntaxe et certaines erreurs d'exécution. Python (et tout autre langage de programmation) ne sait pas automatiquement si votre résultat a du sens physiquement.

---

## 5.3 Vérification manuelle du modèle

Avant d'écrire des tests automatisés, il faut savoir quoi vérifier.

Pour notre modèle de refroidissement, nous pouvons formuler plusieurs propriétés attendues.

### Propriété 1 : température initiale

À `t = 0`, la température doit être égale à la température initiale :

$T(0) = T_0$

### Propriété 2 : refroidissement

Si la sphère est plus chaude que l'environnement, sa température doit diminuer avec le temps.

### Propriété 3 : température ambiante

La température doit se rapprocher de la température ambiante sans passer en dessous dans ce modèle.

### Propriété 4 : influence de `k`

Une valeur plus grande du coefficient de refroidissement doit produire un refroidissement plus rapide.

Ces propriétés ne viennent pas de Python. Elles viennent de notre compréhension du modèle et du problème physique.

### ▶ À faire

Calculons quelques valeurs avec la fonction correcte.

In [ ]:
T0 = 500.0
T_inf = 25.0
k = 0.03

T_t0 = temperature_sphere(0.0, T0, T_inf, k)
T_t50 = temperature_sphere(50.0, T0, T_inf, k)
T_t200 = temperature_sphere(200.0, T0, T_inf, k)

print("T(0 s)   =", T_t0)
print("T(50 s)  =", T_t50)
print("T(200 s) =", T_t200)

### ⚠ Réflexion

Ces trois valeurs sont-elles cohérentes avec un refroidissement ?

- `T(0 s)` est-elle égale à `T0` ?
- `T(50 s)` est-elle plus petite que `T(0 s)` ?
- `T(200 s)` est-elle encore plus petite ?
- Les températures restent-elles au-dessus de `T_inf` ?

Ce sont des vérifications simples, mais elles sont déjà très utiles.

---

## 5.4 Automatiser une vérification avec `assert`

Dans les parties précédentes, nous avons déjà utilisé `assert`. Un `assert` permet de transformer une vérification en instruction Python.

Par exemple :

```python
assert temperature > 0
```

signifie que je m'attends à ce que cette condition soit vraie. Si elle est fausse, le programme doit s'arrêter.

Un test n'est rien d'autre qu'une vérification que vous avez décidé d'automatiser.

### ▶ À faire

Exécutez la cellule suivante.

In [ ]:
assert np.isclose(T_t0, T0), "À t = 0, la température doit être égale à T0."
assert T_t50 < T_t0, "La température devrait diminuer entre 0 s et 50 s."
assert T_t200 < T_t50, "La température devrait continuer à diminuer."
assert T_t200 > T_inf, "La température ne devrait pas passer sous la température ambiante."

print("✅ Toutes les vérifications sont satisfaites")

### 💡 À retenir

Les vérifications précédentes ne démontrent pas que le modèle est parfait. Elles permettent toutefois de détecter rapidement certaines erreurs.

Par exemple :

- un mauvais signe dans l'exponentielle ;
- une mauvaise variable utilisée dans la formule ;
- une confusion entre température initiale et température ambiante.

Les tests ne remplacent pas la réflexion. Ils automatisent une partie de cette réflexion.

### ✍ Exercice 5.1

Complétez les vérifications suivantes pour la fonction correcte `temperature_sphere`.

Nous voulons vérifier que :

1. `T(0) = T0` ;
2. la température après 100 s est inférieure à la température initiale ;
3. la température après 100 s reste supérieure à la température ambiante.

In [ ]:
T0 = 500.0
T_inf = 25.0
k = 0.03

T_0 = temperature_sphere(0.0, T0, T_inf, k)
T_100 = temperature_sphere(100.0, T0, T_inf, k)

assert ???
assert ???
assert ???

print("✅ Exercice 5.1 réussi")

---

## 5.5 Quand une vérification échoue

Voyons ce qui se passe lorsque nous appliquons les mêmes vérifications à la fonction incorrecte.

### ▶ À faire

Exécutez la cellule suivante.

In [ ]:
T_0_faux = temperature_sphere_fausse(0.0, T0, T_inf, k)
T_100_faux = temperature_sphere_fausse(100.0, T0, T_inf, k)

print("T(0 s) avec la fonction fausse   =", T_0_faux)
print("T(100 s) avec la fonction fausse =", T_100_faux)

La fonction fausse peut même passer certaines vérifications.

Par exemple, à `t = 0`, elle donne encore la bonne température initiale.

C'est important : **un seul test ne suffit pas toujours**.

### ▶ À faire

Exécutez la cellule suivante. Une des vérifications devrait échouer.

In [ ]:
assert np.isclose(T_0_faux, T0), "À t = 0, la température doit être égale à T0."
assert T_100_faux < T0, "Après 100 s, la température devrait être inférieure à la température initiale."

print("Si ce message s'affiche, les vérifications ont réussi.")

### ⚠ Réflexion

Lorsque le test échoue, Python affiche un message d'erreur.

Ce n'est pas une mauvaise nouvelle. Au contraire, le test vient de détecter un problème.

Il vaut mieux découvrir une erreur maintenant que plusieurs jours plus tard, après avoir construit toute une analyse sur un résultat faux.

---

## 5.6 Passer de `assert` à `pytest`

Les cellules avec `assert` sont utiles dans un notebook. Mais dans un projet, on souhaite souvent regrouper les tests dans des fichiers séparés.

C'est le rôle de `pytest`. Un test avec `pytest` est simplement une fonction dont le nom commence par `test_`.

Par exemple :

```python
def test_temperature_initiale():
    assert np.isclose(temperature_sphere(0, 500, 25, 0.03), 500)
```

Lorsque vous exécutez `pytest`, Python cherche automatiquement les fonctions qui commencent par `test_` et les exécute.

### ▶ À faire

Dans le notebook, nous pouvons déjà écrire des fonctions de test et les appeler manuellement.

In [ ]:
def test_temperature_initiale():
    T = temperature_sphere(0.0, 500.0, 25.0, 0.03)
    assert np.isclose(T, 500.0)

def test_temperature_diminue():
    T0_calculee = temperature_sphere(0.0, 500.0, 25.0, 0.03)
    T100_calculee = temperature_sphere(100.0, 500.0, 25.0, 0.03)
    assert T100_calculee < T0_calculee

def test_temperature_superieure_ambiante():
    temps_test = np.linspace(0, 200, 201)
    temperatures_test = temperature_sphere(temps_test, 500.0, 25.0, 0.03)
    assert np.all(temperatures_test >= 25.0)

test_temperature_initiale()
test_temperature_diminue()
test_temperature_superieure_ambiante()

print("✅ Tous les tests ont réussi")

### 💡 À retenir

Un test doit être :

- court ;
- explicite ;
- lié à une propriété importante du modèle ;
- facile à comprendre en cas d'échec.

Le nom du test est important. Un nom comme `test_temperature_initiale` est beaucoup plus informatif que `test_1`.

---

## 5.7 Créer un fichier de tests

Dans un vrai projet, les tests seraient placés dans un fichier séparé.

Par exemple :

```text
tests/
└── test_refroidissement.py
```

Le fichier pourrait contenir :

```python
import numpy as np
from refroidissement import temperature_sphere

def test_temperature_initiale():
    T = temperature_sphere(0.0, 500.0, 25.0, 0.03)
    assert np.isclose(T, 500.0)
```

Puis, dans le terminal, on exécuterait :

```bash
pytest
```

ou, pour plus de détails :

```bash
pytest -v
```

Dans ce bootcamp, nous restons principalement dans le notebook, mais l'idée est la même.

### ⚠ Réflexion

Pourquoi séparer les tests du reste du code ?

Parce que cela permet de :

- relancer toutes les vérifications rapidement ;
- vérifier que le code fonctionne encore après une modification ;
- documenter le comportement attendu ;
- détecter les erreurs plus tôt ;
- travailler plus facilement en équipe.

Les tests ne servent pas seulement à trouver des erreurs. Ils décrivent aussi ce que le programme est censé faire.

---

## 5.8 Écrire vos propres tests

Nous allons maintenant écrire quelques tests supplémentaires.

### ✍ Exercice 5.2

Complétez la fonction de test suivante.

Elle doit vérifier qu'une valeur plus grande de `k` donne une température plus basse après 50 s.

In [ ]:
def test_grand_k_refroidit_plus_vite():
    T_k001 = temperature_sphere(50.0, 500.0, 25.0, 0.01)
    T_k008 = temperature_sphere(50.0, 500.0, 25.0, 0.08)

    assert ???

test_grand_k_refroidit_plus_vite()

print("✅ Exercice 5.2 réussi")

### ✍ Exercice 5.3

Complétez la fonction de test suivante.

Elle doit vérifier que si la température initiale est déjà égale à la température ambiante, la température reste constante.

In [ ]:
def test_temperature_constante_si_equilibre():
    temps_test = np.linspace(0, 200, 201)

    temperatures = temperature_sphere(
        temps_test,
        temperature_initiale=25.0,
        temperature_ambiante=25.0,
        coefficient_refroidissement=0.03
    )

    assert ???

test_temperature_constante_si_equilibre()

print("✅ Exercice 5.3 réussi")

### 💡 À retenir

Un bon test vient souvent d'une question physique simple :

- Que devrait-il se passer au temps initial ?
- Que devrait-il se passer à long terme ?
- Que devrait-il se passer si un paramètre augmente ?
- Que devrait-il se passer dans un cas limite ?

Ces questions sont souvent plus importantes que la syntaxe de `pytest`.

---

## 5.9 Les tests facilitent le refactoring

Dans la Partie 4, nous avons vu le concept de _refactoring_.

Refactoriser signifie améliorer la structure du code sans changer son comportement. Mais comment savoir que le comportement n'a pas changé ?

Réponse : en relançant les tests.

### Exemple

Voici une version légèrement réécrite de la fonction.

In [ ]:
def temperature_sphere_refactorisee(t, temperature_initiale, temperature_ambiante, coefficient_refroidissement):
    """
    Même modèle que temperature_sphere, mais écrit avec des variables intermédiaires.
    """
    ecart_initial = temperature_initiale - temperature_ambiante
    facteur_refroidissement = np.exp(-coefficient_refroidissement * t)

    return temperature_ambiante + ecart_initial * facteur_refroidissement

### ▶ À faire

Comparons les deux versions.

In [ ]:
temps_test = np.linspace(0, 200, 201)

T_originale = temperature_sphere(temps_test, 500.0, 25.0, 0.03)
T_refactorisee = temperature_sphere_refactorisee(temps_test, 500.0, 25.0, 0.03)

assert np.allclose(T_originale, T_refactorisee)

print("✅ La version refactorisée donne le même résultat")

### ⚠ Réflexion

Le test précédent ne dit pas que le modèle physique est parfait. Il dit seulement que la version refactorisée se comporte comme la version originale pour le cas testé.

C'est déjà très utile : cela permet de modifier la structure du code avec plus de confiance.

---

## 5.10 Vérification et validation

Les mots **vérification** et **validation** sont souvent confondus. Ils ne signifient pas exactement la même chose.

| Question | Concept |
|---|---|
| Le code résout-il correctement les équations que nous avons écrites ? | Vérification |
| Les équations représentent-elles correctement le système réel ? | Validation |

Dans ce bootcamp, nous nous concentrons surtout sur la vérification.

Par exemple, lorsque nous testons que :

```python
T(0) = T0
```

nous vérifions que le code respecte une propriété du modèle mathématique.

Mais une autre question serait : Le modèle exponentiel représente-t-il fidèlement le refroidissement d'une vraie sphère métallique dans l'air ?

Cette question relève davantage de la validation. Pour y répondre, il faudrait par exemple comparer le modèle à des mesures expérimentales ou à un modèle physique plus détaillé.

### ⚠ Réflexion

Parmi les questions suivantes, lesquelles relèvent plutôt de la vérification ? Lesquelles relèvent plutôt de la validation ?

1. Le code donne-t-il `T0` lorsque `t = 0` ?
2. Le modèle exponentiel correspond-il à des mesures expérimentales ?
3. La température diminue-t-elle lorsque `k` augmente ?
4. Le coefficient `k = 0.03 1/s` est-il réaliste pour une sphère d'acier dans l'air ?
5. Le graphique utilise-t-il les bonnes unités ?

Cette distinction deviendra importante dans la suite du cours.

---

## 5.11 Mini-défi : proposer une vérification

Jusqu'à présent, les vérifications vous ont été données. En pratique, vous devrez souvent proposer vos propres vérifications.

### ✍ Exercice 5.4

Proposez une vérification supplémentaire pour le modèle de refroidissement.

Quelques idées possibles :

- la température reste toujours inférieure ou égale à `T0` ;
- la température reste toujours supérieure ou égale à `T_inf` ;
- si `k = 0`, la température reste constante ;
- si `T0 = T_inf`, la température reste constante ;
- une augmentation de `k` accélère le refroidissement.

Choisissez une idée et complétez la fonction suivante.

In [ ]:
def test_votre_verification():
    # Écrivez ici votre propre vérification.
    # Vous pouvez remplacer entièrement le contenu de cette fonction.

    temps_test = np.linspace(0, 200, 201)
    temperatures = temperature_sphere(temps_test, 500.0, 25.0, 0.03)

    assert ???

test_votre_verification()

print("✅ Exercice 5.4 réussi")

### ⚠ Réflexion

Expliquez brièvement à votre voisin ou au chargé de TD :

- quelle propriété vous avez testée ;
- pourquoi cette propriété devrait être vraie ;
- quel type d'erreur votre test pourrait détecter.

C'est cette réflexion qui est la plus importante.

---

## 5.12 Pourquoi cette partie est importante pour l'IA générative

Un outil d'IA peut générer une fonction Python qui semble correcte. Il peut aussi générer une fonction qui semble correcte mais contient une erreur.

La question essentielle est donc : **Comment savoir si le code généré est fiable ?**

La réponse n'est pas : Parce que l'IA l'a écrit.

La réponse n'est pas non plus : Parce que le code s'exécute.

La réponse est : **Parce que nous avons vérifié le comportement du code.**

Les tests ne servent pas seulement à vérifier votre propre code. Ils servent à vérifier **n'importe quel code**, quelle que soit son origine.

---

## 5.13 Point de contrôle final

À ce stade, vous devriez être capable de :

- expliquer pourquoi un code qui s'exécute peut être faux ;
- formuler des propriétés attendues d'un modèle ;
- écrire une vérification avec `assert` ;
- écrire une fonction de test simple ;
- comprendre le rôle de `pytest` ;
- interpréter un test qui échoue ;
- utiliser les tests pour vérifier un refactoring ;
- distinguer vérification et validation ;
- expliquer pourquoi les tests sont importants lorsque du code est généré par IA.

## 💡 À retenir – Partie 5

Dans cette partie, nous avons vu que la vérification est une partie essentielle de la modélisation numérique.

Les idées importantes sont :

- Un code qui s'exécute n'est pas nécessairement correct.
- Les graphiques sont utiles, mais ils ne suffisent pas toujours.
- Une vérification commence par une propriété attendue du modèle.
- `assert` permet d'automatiser une vérification simple.
- `pytest` permet d'organiser et d'exécuter des tests automatiquement.
- Un test qui échoue est une information utile.
- Les tests facilitent le refactoring.
- La vérification concerne le code et les équations.
- La validation concerne l'adéquation du modèle avec la réalité.
- Les tests sont essentiels pour évaluer du code, peu importe son origine.

Phrase centrale : **La responsabilité de vérifier un résultat appartient toujours à l'ingénieur, peu importe l'origine du code !**

🚀 Dans la **Partie 6 – Utilisation responsable de l'IA générative**, nous verrons comment appliquer cette idée aux outils d'aide à la programmation.